# AASN Phase Transition Experiment

Runs the context propagation phase transition experiment on Colab.
Results are saved to Google Drive so they persist across sessions.

**Setup:** ~5 minutes | **Runtime:** ~15 hours for full matrix

Safe to disconnect and reconnect — results checkpoint after each run.

## 1. Mount Google Drive (persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create results directory on Drive
!mkdir -p /content/drive/MyDrive/aasn_experiments/results
print('Drive mounted. Results will be saved to: /content/drive/MyDrive/aasn_experiments/results')

## 2. Clone repo and install dependencies

In [ ]:
# Clone the repo
!git clone https://github.com/jemsbhai/aasn.git /content/aasn
%cd /content/aasn

# Install experiment dependencies
!pip install -q openai pydantic numpy sentence-transformers pyyaml matplotlib pandas scipy datasets huggingface-hub

## 3. Install and start Ollama

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
import subprocess
import time

ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)  # Wait for server to start
print(f'Ollama server started (PID: {ollama_proc.pid})')

# Verify it's running
!curl -s http://localhost:11434/api/tags | python3 -c "import sys,json; print('Ollama OK:', json.load(sys.stdin).get('models', []))"

In [ ]:
# Pull the model (~4.7GB, takes a few minutes)
!ollama pull qwen2.5-coder:7b
print('\nModel ready.')
!ollama list

## 4. Download benchmarks

In [ ]:
%cd /content/aasn/experiments/context_propagation
!python download_benchmarks_cli.py --tier 1

## 5. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'\nCUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 6. Symlink results to Google Drive

This ensures all results write directly to Drive, surviving disconnects.

In [ ]:
import os

# Remove local results dir if it exists and symlink to Drive
local_results = '/content/aasn/experiments/context_propagation/results'
drive_results = '/content/drive/MyDrive/aasn_experiments/results'

if os.path.islink(local_results):
    os.unlink(local_results)
elif os.path.isdir(local_results):
    # Move any existing results to Drive first
    !cp -rn {local_results}/* {drive_results}/ 2>/dev/null || true
    !rm -rf {local_results}

os.symlink(drive_results, local_results)
print(f'Results symlinked: {local_results} -> {drive_results}')
print(f'All results will persist on Google Drive.')

## 7. Run experiment: Rich Feedback

10 runs x 50 generations x 50 tasks. ~7-8 hours.

If disconnected, reconnect and run cell 8 (Resume) instead of this cell.

In [ ]:
%cd /content/aasn/experiments/context_propagation
!python run_experiment.py --config configs/phase_transition_rich.yaml

## 8. Resume (if disconnected)

Run cells 1-3 first (mount drive, clone repo, start ollama), then this cell.
It picks up from the last completed run.

In [ ]:
# Re-setup after reconnect
import subprocess, time, os

# Restart Ollama if needed
try:
    import urllib.request
    urllib.request.urlopen('http://localhost:11434/api/tags')
    print('Ollama already running.')
except:
    print('Starting Ollama...')
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    print('Ollama started.')

# Re-symlink results to Drive
local_results = '/content/aasn/experiments/context_propagation/results'
drive_results = '/content/drive/MyDrive/aasn_experiments/results'
if not os.path.islink(local_results):
    if os.path.isdir(local_results):
        !cp -rn {local_results}/* {drive_results}/ 2>/dev/null || true
        !rm -rf {local_results}
    os.symlink(drive_results, local_results)
    print('Results re-symlinked to Drive.')

# Re-download benchmarks if needed
%cd /content/aasn/experiments/context_propagation
if not os.path.exists('benchmarks/coding/humaneval.json'):
    !python download_benchmarks_cli.py --tier 1
else:
    print('Benchmarks already downloaded.')

print('\nReady to resume.')

In [ ]:
# Resume rich feedback experiment
%cd /content/aasn/experiments/context_propagation
!python run_experiment.py --config configs/phase_transition_rich.yaml --resume

In [ ]:
# Resume score-only experiment
%cd /content/aasn/experiments/context_propagation
!python run_experiment.py --config configs/phase_transition_score_only.yaml --resume

## 9. Run experiment: Score-Only Control

Run after rich feedback completes.

In [ ]:
%cd /content/aasn/experiments/context_propagation
!python run_experiment.py --config configs/phase_transition_score_only.yaml

## 10. Check progress (run anytime)

In [ ]:
import json
from pathlib import Path

results_dir = Path('/content/aasn/experiments/context_propagation/results')

for experiment_dir in sorted(results_dir.iterdir()):
    if not experiment_dir.is_dir():
        continue
    print(f'\n=== {experiment_dir.name} ===')
    for ts_dir in sorted(experiment_dir.iterdir()):
        if not ts_dir.is_dir():
            continue
        cp_path = ts_dir / 'checkpoint.json'
        if cp_path.exists():
            cp = json.loads(cp_path.read_text())
            status = cp.get('status', 'in_progress')
            print(f'  {ts_dir.name}: {cp["completed_runs"]}/{cp["total_runs"]} runs ({status})')
        else:
            runs = len(list(ts_dir.glob('run_*')))
            print(f'  {ts_dir.name}: {runs} run dirs (no checkpoint)')

        # Show aggregate if complete
        agg_path = ts_dir / 'aggregate_results.json'
        if agg_path.exists():
            agg = json.loads(agg_path.read_text())
            r = agg.get('results', {})
            print(f'    fidelity: {r.get("avg_final_fidelity", 0):.3f} +/- {r.get("std_final_fidelity", 0):.3f}')
            print(f'    rot_rate: {r.get("avg_rot_rate", 0):.4f} +/- {r.get("std_rot_rate", 0):.4f}')
            print(f'    drift:    {r.get("avg_total_drift", 0):.3f} +/- {r.get("std_total_drift", 0):.3f}')

## 11. Quick visualization (run after experiments complete)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def load_trajectories(experiment_name):
    """Load score trajectories from all runs of an experiment."""
    base = Path('/content/aasn/experiments/context_propagation/results') / experiment_name
    if not base.exists():
        return None
    # Find the latest timestamped directory
    ts_dirs = sorted([d for d in base.iterdir() if d.is_dir()])
    if not ts_dirs:
        return None
    latest = ts_dirs[-1]
    trajectories = []
    for run_dir in sorted(latest.glob('run_*')):
        summary_path = run_dir / 'chain_summary.json'
        if summary_path.exists():
            summary = json.loads(summary_path.read_text())
            scores = [g['avg_score'] for g in summary['score_trajectory']]
            trajectories.append(scores)
    return trajectories

rich = load_trajectories('qwen7b_rich_50gen')
score_only = load_trajectories('qwen7b_score_only_50gen')

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

for label, data, color in [
    ('Rich Feedback', rich, 'blue'),
    ('Score-Only', score_only, 'red'),
]:
    if data is None:
        continue
    arr = np.array(data)
    mean = arr.mean(axis=0)
    std = arr.std(axis=0)
    gens = np.arange(len(mean))
    ax.plot(gens, mean, label=f'{label} (n={len(data)})', color=color)
    ax.fill_between(gens, mean - std, mean + std, alpha=0.2, color=color)

ax.set_xlabel('Generation')
ax.set_ylabel('Average Score')
ax.set_title('Phase Transition: Rich Feedback vs Score-Only (Qwen2.5-Coder-7B)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/aasn_experiments/phase_transition_plot.png', dpi=150)
plt.show()
print('Plot saved to Google Drive.')